# Average Properties Extraction Results

In [2]:
import os
import sys

import numpy as np
import pandas as pd

# Get the current working directory of the notebook
notebook_dir = os.getcwd()
# print(notebook_dir)
# Add the parent directory to the system path
sys.path.append(os.path.join(notebook_dir, '../'))

from metrics import EvaluationMetric
from data_processing import DataProcessing

In [ ]:
models = ["openai_gpt-oss-120b", "openai_gpt-oss-20b", "llama-3.3-70b-versatile"]

## Load Data

In [33]:
base_data_path = DataProcessing.load_base_data_path(notebook_dir)
dataset_folder = os.path.join(base_data_path, "classification_results/synthetic-fpb-chronicle2050-yt-news-timebank-mf_climate/extract_properties/classification/")

In [34]:
seed_files = os.listdir(dataset_folder)
seed_files

['seed3', 'seed7', 'seed33']

In [35]:
model_files_to_avg = []

for seed_file in seed_files:
    print(f"Seed: {seed_file}")
    models_folders = os.path.join(dataset_folder, seed_file)
    model_files = os.listdir(models_folders)

    for model_file in model_files:
        print(f"  Model: {model_file}")
        model_path = os.path.join(models_folders, model_file)
        # print(f"  Model Path: {model_path}")
        data_files = os.listdir(model_path)

        for data_file in data_files:
            if 'metrics_summary' in data_file and '.csv' in data_file:
                print(f"    File: {data_file}")
                metrics_summary_file_path = os.path.join(models_folders, model_file, data_file)

                df = DataProcessing.load_from_file(metrics_summary_file_path, 'csv', sep=',')

                model_data = {
                    'seed':                      seed_file,
                    'model_name':                model_file,
                    'metrics_summary_file':      data_file,
                    'metrics_summary_file_path': metrics_summary_file_path,
                    'df':                        df
                }
                model_files_to_avg.append(model_data)
    print()

Seed: seed3
  Model: openai_gpt-oss-120b
    File: metrics_summary_openai_gpt-oss-120b-v1.csv
  Model: openai_gpt-oss-20b
    File: metrics_summary_openai_gpt-oss-20b-v1.csv
  Model: llama-3.3-70b-versatile
    File: metrics_summary_llama-3.3-70b-versatile-v1.csv

Seed: seed7
  Model: openai_gpt-oss-120b
    File: metrics_summary_openai_gpt-oss-120b-v1.csv
  Model: openai_gpt-oss-20b
    File: metrics_summary_openai_gpt-oss-20b-v1.csv
  Model: llama-3.3-70b-versatile
    File: metrics_summary_llama-3.3-70b-versatile-v1.csv

Seed: seed33
  Model: openai_gpt-oss-120b
    File: metrics_summary_openai_gpt-oss-120b-v1.csv
  Model: openai_gpt-oss-20b
    File: metrics_summary_openai_gpt-oss-20b-v1.csv
  Model: llama-3.3-70b-versatile
    File: metrics_summary_llama-3.3-70b-versatile-v1.csv



In [36]:
# -------------------------------------------------------
# Group by model_name, then average across seeds
# -------------------------------------------------------
# { model_name: [df_seed3, df_seed7, df_seed33] }
grouped_by_model = {}

for model_data in model_files_to_avg:
    model_name = model_data['model_name']
    if model_name not in grouped_by_model:
        grouped_by_model[model_name] = []
    grouped_by_model[model_name].append(model_data['df'])

print("\n" + "="*60)
print("AVERAGED RESULTS PER MODEL ACROSS SEEDS")
print("="*60)

averaged_results = {}

for model_name, dfs in grouped_by_model.items():
    print(f"\nModel: {model_name} | Seeds: {len(dfs)}")

    # Stack all seed DataFrames
    combined_df = pd.concat(dfs, ignore_index=True)

    # Average numeric columns, grouped by property
    # so Source, Target, Date, Outcome are averaged separately
    numeric_cols = combined_df.select_dtypes(include=[np.number]).columns.tolist()

    mean_df = combined_df.groupby('property')[numeric_cols].mean().reset_index()
    std_df  = combined_df.groupby('property')[numeric_cols].std().reset_index()

    mean_df.insert(0, 'model', model_name)
    std_df.insert(0,  'model', model_name)

    print(mean_df)
    averaged_results[model_name] = {
        'mean': mean_df,
        'std':  std_df
    }


AVERAGED RESULTS PER MODEL ACROSS SEEDS

Model: openai_gpt-oss-120b | Seeds: 3
                 model     property       seed  test_accuracy  \
0  openai_gpt-oss-120b         Date  14.333333       0.714286   
1  openai_gpt-oss-120b  No Property  14.333333       0.190476   
2  openai_gpt-oss-120b      Outcome  14.333333       0.285714   
3  openai_gpt-oss-120b       Source  14.333333       0.380952   
4  openai_gpt-oss-120b       Target  14.333333       0.285714   

   precision_class_0  precision_class_1  recall_class_0  recall_class_1  \
0           0.793651           0.333333        0.904762        0.111111   
1           0.000000           0.666667        0.000000        0.190476   
2           0.373016           0.000000        0.688889        0.000000   
3           0.400000           0.333333        0.588889        0.150000   
4           0.194444           0.555556        0.416667        0.177778   

   f1_class_0  f1_class_1        tn        fp        fn        tp  
0    0.822

In [37]:
# -------------------------------------------------------
# Save averaged results per model
# -------------------------------------------------------
avg_save_path = os.path.join(
    base_data_path,
    'classification_results',
    'synthetic-fpb-chronicle2050-yt-news-timebank-mf_climate',
    'extract_properties',
    'classification',
    'average'
)
os.makedirs(avg_save_path, exist_ok=True)

for model_name, results in averaged_results.items():
    mean_df = results['mean']
    std_df  = results['std']

    # Replace "/" with "_" to avoid nested folders in filename
    # e.g., "openai/gpt-oss-120b" -> "openai_gpt-oss-120b"
    clean_model_name = model_name.replace('/', '_')

    DataProcessing.save_to_file(
        data=mean_df,
        path=avg_save_path,
        prefix=f'mean_{clean_model_name}',
        save_file_type='csv',
        include_version=False
    )

    DataProcessing.save_to_file(
        data=std_df,
        path=avg_save_path,
        prefix=f'std_{clean_model_name}',
        save_file_type='csv',
        include_version=False
    )

    print(f"✓ Saved averaged results for: {model_name}")

print(f"\n✓ All averaged results saved to: {avg_save_path}")

Saving CSV file to: /Users/detraviousjamaribrinkley/Documents/Development/research_labs/uf_ds/predictions/properties_extraction_experiments/../data/classification_results/synthetic-fpb-chronicle2050-yt-news-timebank-mf_climate/extract_properties/classification/average/mean_openai_gpt-oss-120b.csv
Saving CSV file to: /Users/detraviousjamaribrinkley/Documents/Development/research_labs/uf_ds/predictions/properties_extraction_experiments/../data/classification_results/synthetic-fpb-chronicle2050-yt-news-timebank-mf_climate/extract_properties/classification/average/std_openai_gpt-oss-120b.csv
✓ Saved averaged results for: openai_gpt-oss-120b
Saving CSV file to: /Users/detraviousjamaribrinkley/Documents/Development/research_labs/uf_ds/predictions/properties_extraction_experiments/../data/classification_results/synthetic-fpb-chronicle2050-yt-news-timebank-mf_climate/extract_properties/classification/average/mean_openai_gpt-oss-20b.csv
Saving CSV file to: /Users/detraviousjamaribrinkley/Docume

In [54]:
# -------------------------------------------------------
# Combine all averaged results into one table
# -------------------------------------------------------
results_dfs = []
for model_name, results in averaged_results.items():
    mean_df = results['mean']
    std_df  = results['std']
    results_dfs.append(mean_df)

result_df = DataProcessing.concat_dfs(results_dfs)
result_df = result_df.reset_index(drop=True)

print("\n" + "="*60)
print("COMBINED AVERAGED RESULTS — ALL MODELS")
print("="*60)
print(result_df)

# Save combined CSV
DataProcessing.save_to_file(
    data=result_df,
    path=avg_save_path,
    prefix='mean_all_models',
    save_file_type='csv',
    include_version=False
)
print(f"\n✓ Saved combined averaged results to: {avg_save_path}/mean_all_models.csv")

# -------------------------------------------------------
# Export to LaTeX
# -------------------------------------------------------

# Select key columns for LaTeX table
# Adjust based on what columns matter most for your paper
latex_cols = [
    'model',
    'property',
    'test_accuracy',
    'precision_class_0',
    'precision_class_1',
    'recall_class_0',
    'recall_class_1',
    'f1_class_0',
    'f1_class_1',
    'tn', 'fp', 'fn', 'tp'
]

# Only keep columns that exist
available_latex_cols = [col for col in latex_cols if col in result_df.columns]
latex_df = result_df[available_latex_cols].copy()

# Round to 4 decimal places for cleaner LaTeX output
numeric_cols = latex_df.select_dtypes(include=[np.number]).columns
latex_df[numeric_cols] = latex_df[numeric_cols].round(4)

# Generate LaTeX string
latex_str = latex_df.to_latex(
    index=False,
    escape=False,
    float_format="%.4f",
    caption="Averaged Property Extraction Results Across Seeds",
    label="tab:property_extraction_results"
)

print("\n" + "="*60)
print("LATEX TABLE")
print("="*60)
print(latex_str)

# Save LaTeX to file
# latex_save_path = os.path.join(avg_save_path, 'mean_all_models.tex')
# with open(latex_save_path, 'w') as f:
#     f.write(latex_str)

# print(f"\n✓ Saved LaTeX table to: {latex_save_path}")


COMBINED AVERAGED RESULTS — ALL MODELS
                      model     property       seed  test_accuracy  \
0       openai_gpt-oss-120b         Date  14.333333       0.714286   
1       openai_gpt-oss-120b  No Property  14.333333       0.190476   
2       openai_gpt-oss-120b      Outcome  14.333333       0.285714   
3       openai_gpt-oss-120b       Source  14.333333       0.380952   
4       openai_gpt-oss-120b       Target  14.333333       0.285714   
5        openai_gpt-oss-20b         Date  14.333333       0.761905   
6        openai_gpt-oss-20b  No Property  14.333333       0.142857   
7        openai_gpt-oss-20b      Outcome  14.333333       0.428571   
8        openai_gpt-oss-20b       Source  14.333333       0.428571   
9        openai_gpt-oss-20b       Target  14.333333       0.190476   
10  llama-3.3-70b-versatile         Date  14.333333       0.761905   
11  llama-3.3-70b-versatile  No Property  14.333333       0.238095   
12  llama-3.3-70b-versatile      Outcome  14.33333